# Seminar 5 – Programarea algoritmilor

- complexitati de timp (Big-O)
- algoritmi de tip **divide et impera**
- probleme clasice de algoritmica:
  - cautare binara in array rotit
  - exponentiere rapida
  - subarray cu suma k (k-sum subarray)
  - cel mai lung substring cu caractere distincte
  - alte probleme clasice (maximum subarray, majority element, 2-sum, 3-sum)

## 1. Recap: Complexitate de timp (Big-O)

### Exemplul 1 – Bucla imbricata cu impartire la 2

Analizam urmatoarea functie:
```python
def f(n: int) -> int:
    s = 0
    for i in range(1, n + 1):
        j = i
        while j > 0:
            s += 1
            j //= 2
    return s
```

- Pentru un `i` fix, bucla `while` imparte `j` la 2 pana ajunge la 0, deci are ~ `log2(i)` pasi.
- Numarul total de operatii este aproximativ `sum_{i=1..n} log i ≈ n log n`.
- Concluzie: **complexitatea este `O(n log n)`**.


In [1]:
# Exemplu: numar de pasi pentru functia f
def f(n: int) -> int:
    s = 0
    for i in range(1, n + 1):
        j = i
        while j > 0:
            s += 1
            j //= 2
    return s

for n in [10, 100, 1000, 10000]:
    print(f"n={n}, s={f(n)}")

n=10, s=29
n=100, s=580
n=1000, s=8987
n=10000, s=123631


### Exemplul 2 – Functie recursiva si recurenta

```python
def g(n: int) -> int:
    if n <= 1:
        return 1
    return g(n // 2) + g(n // 2) + n
```

Scriem recurenta pentru timpul de executie `T(n)`:
- `T(n) = 2 · T(n/2) + O(n)`, cu `T(1) = O(1)`.

Din Master Theorem rezulta `T(n) = Θ(n log n)`.

In [2]:
# Exemplu simplu de verificare a cresterii timpului pentru g(n)
def g(n: int) -> int:
    if n <= 1:
        return 1
    return g(n // 2) + g(n // 2) + n

print(g(8))  # doar ca sa vedem ca functia merge

32


### Exemplul 3 – Cautare binara clasica

```python
def binary_search(a, x):
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if a[mid] == x:
            return mid
        elif a[mid] < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1
```

La fiecare pas injumatatim intervalul de cautare.
- Recurenta: `T(n) = T(n/2) + O(1)`.
- Complexitate: **`O(log n)`**.


In [3]:
# Implementare binary search si teste simple
def binary_search(a, x):
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if a[mid] == x:
            return mid
        elif a[mid] < x:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

arr = [1, 3, 5, 7, 9, 11]
for x in [1, 4, 11, 12]:
    print(x, '->', binary_search(arr, x))

1 -> 0
4 -> -1
11 -> 5
12 -> -1


## 2. Divide et impera: Cautare in array rotit

Consideram un vector sortat crescator, dar apoi **rotit** de un numar necunoscut de ori:
```text
Original: [1, 2, 3, 4, 5, 6, 7]
Rotit:    [4, 5, 6, 7, 1, 2, 3]
```

Ideea principala: pentru un `mid` dat, unul dintre segmentele `[lo..mid]` sau `[mid..hi]` este in continuare sortat.
Folosim acest lucru pentru a decide in care jumatate cautam in continuare, pastrand complexitatea `O(log n)`.

In [4]:
# Cautare intr-un array sortat si rotit in O(log n)
def search_rotated(a, x):
    lo, hi = 0, len(a) - 1
    while lo <= hi:
        mid = (lo + hi) // 2
        if a[mid] == x:
            return mid

        # Partea stanga [lo..mid] este sortata
        if a[lo] <= a[mid]:
            if a[lo] <= x < a[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        else:
            # Partea dreapta [mid..hi] este sortata
            if a[mid] < x <= a[hi]:
                lo = mid + 1
            else:
                hi = mid - 1

    return -1

arr = [4, 5, 6, 7, 1, 2, 3]
for x in [1, 3, 4, 7, 8]:
    print(x, '->', search_rotated(arr, x))

1 -> 4
3 -> 6
4 -> 0
7 -> 3
8 -> -1


### Minim intr-un array sortat si rotit

Vrem sa gasim **valoarea minima** in timp `O(log n)`.

Ideea:
- Daca `a[lo] <= a[hi]`, vectorul este deja sortat fara rotatie ⇒ minimul este `a[lo]`.
- Altfel, ne uitam la `mid` si decidem daca minimul este in stanga sau in dreapta.


In [5]:
# Gasirea minimului intr-un array sortat si rotit
def find_min_rotated(a):
    lo, hi = 0, len(a) - 1

    while lo < hi:
        if a[lo] <= a[hi]:
            return a[lo]

        mid = (lo + hi) // 2
        if a[mid] >= a[lo]:
            lo = mid + 1
        else:
            hi = mid

    return a[lo]

print(find_min_rotated([4, 5, 6, 7, 1, 2, 3]))
print(find_min_rotated([1, 2, 3, 4]))

1
1


## 3. Exponentiere rapida (fast exponentiation)

Vrem sa calculam `a^n` in timp `O(log n)` folosind divide et impera.

Ideea:
- `a^n = (a^{n/2})^2` daca n este par;
- `a^n = (a^{(n//2)})^2 * a` daca n este impar.

Recurenta: `T(n) = T(n/2) + O(1)` ⇒ `O(log n)`.

In [6]:
# Exponentiere rapida
def pow_fast(a: int, n: int) -> int:
    if n == 0:
        return 1
    half = pow_fast(a, n // 2)
    if n % 2 == 0:
        return half * half
    else:
        return half * half * a

for n in [0, 1, 2, 5, 10]:
    print(f"2^{n} = {pow_fast(2, n)}")

2^0 = 1
2^1 = 2
2^2 = 4
2^5 = 32
2^10 = 1024


## 4. Problema k-sum subarray

Avem un vector de intregi (pozitive, zero, negative) `a[0..n-1]` si un intreg `k`.
Vrem sa verificam:
1. **Exista** un subarray continuu cu suma exact `k`?
2. Care este **numarul** de astfel de subarray-uri?

Folosim sume prefixate si un set / dictionar pentru a obtine complexitate **`O(n)`**.

In [7]:
# 4.1 Exista subarray cu suma k?
def has_subarray_sum_k(a, k: int) -> bool:
    seen = {0}  # prefix "gol"
    prefix = 0

    for x in a:
        prefix += x
        if prefix - k in seen:
            return True
        seen.add(prefix)
    return False

print(has_subarray_sum_k([1, 2, 3], 3))      # True (subarray [1,2] sau [3])
print(has_subarray_sum_k([1, -1, 2, 3], 4))  # True
print(has_subarray_sum_k([2, 2, 2], 5))      # False


True
True
False


In [8]:
# 4.2 Numarul de subarray-uri cu suma k
from collections import defaultdict

def count_subarrays_sum_k(a, k: int) -> int:
    freq = defaultdict(int)
    freq[0] = 1  # prefix gol
    prefix = 0
    ans = 0

    for x in a:
        prefix += x
        ans += freq[prefix - k]
        freq[prefix] += 1
    return ans

print(count_subarrays_sum_k([1, 1, 1], 2))    # 2 ([1,1] de doua ori)
print(count_subarrays_sum_k([1, 2, 3], 3))    # 2 ([1,2], [3])
print(count_subarrays_sum_k([1, -1, 1, -1], 0))

2
2
4


## 5. Longest distinct substring

Problema:
- Se da un sir `s`.
- Vrem lungimea celui mai lung substring (secventa de caractere **consecutive**) care contine doar **caractere distincte**.

Exemple:
- `"abcabcbb"` → 3 (`"abc"`)
- `"bbbbb"` → 1 (`"b"`)
- `"pwwkew"` → 3 (`"wke"`)

Solutie in `O(n)` cu tehnica **sliding window**:

In [9]:
# Longest distinct substring - solutie O(n)
def longest_distinct_substring(s: str) -> int:
    last = {}  # caracter -> ultima pozitie
    best = 0
    l = 0

    for r, c in enumerate(s):
        if c in last and last[c] >= l:
            l = last[c] + 1
        last[c] = r
        best = max(best, r - l + 1)
    return best

for s in ["abcabcbb", "bbbbb", "pwwkew", "", "abba"]:
    print(repr(s), '->', longest_distinct_substring(s))

'abcabcbb' -> 3
'bbbbb' -> 1
'pwwkew' -> 3
'' -> 0
'abba' -> 2


## 6. Exercitii (de rezolvat)

1. Functie care calculeaza in mod naiv daca exista subarray cu suma k in `O(n^2)` si o comparatie 
   empirica cu varianta `O(n)` pe inputuri mici.
2. Varianta recursiva pentru binary search.
3. Functie care gaseste **indicele de rotatie** (pivotul) intr-un array sortat si rotit.
4. Functie care gaseste **cel mai lung subarray cu suma <= k** pentru numere ne-negative (sliding window).
5. Functie care gaseste **cel mai lung substring cu cel mult k caractere distincte**.


In [10]:
# Exercitiul 1: subarray cu suma k in O(n^2)
def has_subarray_sum_k_brut(a, k: int) -> bool:
    # TODO: implementati varianta O(n^2)
    # hint: doua for-uri imbricate, suma pe [i..j]
    pass

# Exercitiul 2: binary search recursiv
def binary_search_rec(a, x, lo=0, hi=None):
    # TODO: implementati cautarea binara recursiva
    pass

# Exercitiul 3: gasirea pivotului (indicele minimului) intr-un array rotit
def find_rotation_index(a):
    # TODO: folositi idee similara cu find_min_rotated, dar intoarceti indicele, nu valoarea
    pass

# Exercitiul 4: cel mai lung subarray cu suma <= k (pentru a[i] >= 0)
def longest_subarray_sum_leq_k(a, k: int) -> int:
    # TODO: folositi sliding window
    pass

# Exercitiul 5: cel mai lung substring cu cel mult k caractere distincte
def longest_substring_at_most_k_distinct(s: str, k: int) -> int:
    # TODO: generalizati ideea de la longest_distinct_substring
    pass


## 7. Exercitii suplimentare (cu solutii)

In aceasta sectiune sunt variante rezolvate pentru exercitii similare cu cele de mai sus.
Le puteti folosi ca model / corectie.


In [11]:
# Solutie pentru: subarray cu suma k in O(n^2)
def has_subarray_sum_k_brut(a, k: int) -> bool:
    n = len(a)
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += a[j]
            if s == k:
                return True
    return False

print('Brut vs O(n):')
arr = [1, 2, 3, -1, 2]
for k in [3, 4, 100]:
    print('k =', k,
          'brut =', has_subarray_sum_k_brut(arr, k),
          'hash =', has_subarray_sum_k(arr, k))

Brut vs O(n):
k = 3 brut = True hash = True
k = 4 brut = True hash = True
k = 100 brut = False hash = False


In [12]:
# Solutie pentru: binary search recursiv
def binary_search_rec(a, x, lo=0, hi=None):
    if hi is None:
        hi = len(a) - 1
    if lo > hi:
        return -1
    mid = (lo + hi) // 2
    if a[mid] == x:
        return mid
    elif a[mid] < x:
        return binary_search_rec(a, x, mid + 1, hi)
    else:
        return binary_search_rec(a, x, lo, mid - 1)

arr = [1, 3, 5, 7, 9]
for x in [1, 2, 9, 10]:
    print(x, '->', binary_search_rec(arr, x))

1 -> 0
2 -> -1
9 -> 4
10 -> -1


In [13]:
# Solutie pentru: gasirea pivotului (indicele minimului) intr-un array rotit
def find_rotation_index(a):
    lo, hi = 0, len(a) - 1
    while lo < hi:
        if a[lo] <= a[hi]:
            return lo
        mid = (lo + hi) // 2
        if a[mid] >= a[lo]:
            lo = mid + 1
        else:
            hi = mid
    return lo

print(find_rotation_index([4, 5, 6, 7, 1, 2, 3]))  # 4
print(find_rotation_index([1, 2, 3, 4]))            # 0

4
0


In [14]:
# Solutie pentru: cel mai lung subarray cu suma <= k (a[i] >= 0)
def longest_subarray_sum_leq_k(a, k: int) -> int:
    n = len(a)
    l = 0
    s = 0
    best = 0
    for r in range(n):
        s += a[r]
        while s > k and l <= r:
            s -= a[l]
            l += 1
        best = max(best, r - l + 1)
    return best

print(longest_subarray_sum_leq_k([1, 2, 3, 4], 5))   # 2
print(longest_subarray_sum_leq_k([2, 2, 2], 3))      # 1
print(longest_subarray_sum_leq_k([1, 1, 1, 1], 10))  # 4

2
1
4


In [15]:
# Solutie pentru: cel mai lung substring cu cel mult k caractere distincte
def longest_substring_at_most_k_distinct(s: str, k: int) -> int:
    if k <= 0:
        return 0
    freq = {}
    l = 0
    best = 0
    distinct = 0

    for r, c in enumerate(s):
        if c not in freq or freq[c] == 0:
            distinct += 1
        freq[c] = freq.get(c, 0) + 1

        while distinct > k:
            c_left = s[l]
            freq[c_left] -= 1
            if freq[c_left] == 0:
                distinct -= 1
            l += 1

        best = max(best, r - l + 1)
    return best

print(longest_substring_at_most_k_distinct("eceba", 2))  # 3 ("ece")
print(longest_substring_at_most_k_distinct("aa", 1))     # 2
print(longest_substring_at_most_k_distinct("abcabc", 2))

3
2
2


## 8. Alte probleme de algoritmica si discutii de complexitate

In aceasta sectiune adaugam cateva probleme clasice suplimentare si discutam complexitatea lor:
- **Maximum subarray sum** (Kadane vs O(n²))
- **Majority element** (divide et impera vs Boyer–Moore)
- **2-sum si 3-sum** (combinatorica si sortare)
- O mica comparatie intre tipuri de complexitati: `O(n)`, `O(n log n)`, `O(n^2)`, `O(2^n)`.


### 8.1 Maximum subarray sum (Kadane)

Problema:
- Avem un vector de intregi (pozitivi/negativi).
- Vrem **suma maxima** a unui subarray **continuu**.

Solutie naiva: `O(n^2)`
- luam toate perechile `(i, j)` si calculam suma pe `a[i..j]`.

Solutie optima: **Kadane** – `O(n)`
- parcurgem o singura data tabloul,
- mentinem `best_ending_here` (cea mai buna suma care se termina in i),
- si `best_overall` (cea mai buna suma gasita pana acum).


In [16]:
# Maximum subarray sum – Kadane O(n)
def max_subarray_sum(a):
    best_ending_here = a[0]
    best_overall = a[0]
    for x in a[1:]:
        # ori continuam subarray-ul, ori incepem unul nou de la x
        best_ending_here = max(x, best_ending_here + x)
        best_overall = max(best_overall, best_ending_here)
    return best_overall

print(max_subarray_sum([-2,1,-3,4,-1,2,1,-5,4]))  # 6 (subarray [4,-1,2,1])

6


### 8.2 Majority element

Problema:
- Avem un vector `a[0..n-1]`.
- Un element este *majority* daca apare de **strict mai mult de n/2 ori**.

Solutie divide et impera (idee):
- impartim vectorul in doua jumatati,
- aflam candidatele de majority in stanga si dreapta,
- numaram aparitiile si decidem care (daca exista) este majority.
- complexitate: `T(n) = 2T(n/2) + O(n)` ⇒ `O(n log n)`.

Solutie in `O(n)` timp si `O(1)` memorie: **Boyer–Moore voting algorithm**.

In [17]:
# Majority element – Boyer–Moore O(n)
def majority_element(a):
    # 1. Gasim un candidat
    candidate = None
    count = 0
    for x in a:
        if count == 0:
            candidate = x
            count = 1
        elif x == candidate:
            count += 1
        else:
            count -= 1

    # 2. Verificam daca este intr-adevar majority
    if a.count(candidate) > len(a) // 2:
        return candidate
    return None

print(majority_element([2,2,1,1,1,2,2]))  # 2
print(majority_element([1,2,3]))          # None

2
None


### 8.3 2-sum si 3-sum

#### Problema 2-sum
Avem un vector `a` si un numar `k`. Exista doi indici `i < j` cu `a[i] + a[j] = k`?

- Solutie naiva: doua for-uri imbricate ⇒ `O(n^2)`.
- Solutie cu **hash set**: parcurgem o data vectorul, pentru fiecare `x` verificam daca `k - x` a mai aparut ⇒ `O(n)` timp, `O(n)` memorie.

#### Problema 3-sum
Exista trei indici `i < j < k` cu `a[i] + a[j] + a[k] = 0` (sau o alta valoare)?

- Solutie clasica: sortam `a` (`O(n log n)`) si apoi pentru fiecare `i` facem un 2-sum pe `a[i+1..]` cu doi pointeri ⇒ `O(n^2)` timp total.


In [18]:
# 2-sum in O(n) cu hash set
def two_sum(a, k):
    seen = set()
    for x in a:
        if k - x in seen:
            return True
        seen.add(x)
    return False

print(two_sum([2, 7, 11, 15], 9))   # True (2+7)
print(two_sum([1, 2, 3], 10))       # False

True
False


In [19]:
# 3-sum clasic (intoarce toate tripletele cu suma 0)
def three_sum_zero(nums):
    nums = sorted(nums)
    n = len(nums)
    res = []
    for i in range(n):
        if i > 0 and nums[i] == nums[i-1]:
            continue  # evitam duplicatele
        target = -nums[i]
        l, r = i + 1, n - 1
        while l < r:
            s = nums[l] + nums[r]
            if s == target:
                res.append((nums[i], nums[l], nums[r]))
                l += 1
                r -= 1
                while l < r and nums[l] == nums[l-1]:
                    l += 1
                while l < r and nums[r] == nums[r+1]:
                    r -= 1
            elif s < target:
                l += 1
            else:
                r -= 1
    return res

print(three_sum_zero([-1,0,1,2,-1,-4]))

[(-1, -1, 2), (-1, 0, 1)]


### 8.4 Comparatie intre O(n), O(n log n), O(n²), O(2ⁿ)

Un mic experiment: presupunem ca o operatie dureaza 1 unitate de timp si calculam cate operatii ar face
un algoritm cu urmatoarele complexitati, pentru valori diferite ale lui `n`.

Scopul este sa vedem `ordinea de marime`, nu valori exacte.


In [20]:
import math

def approx_ops(n):
    return {
        'O(n)': n,
        'O(n log n)': int(n * math.log2(n) if n > 0 else 0),
        'O(n^2)': n * n,
        'O(2^n)': 2 ** n if n <= 30 else None,  # evitam valori astronomice
    }

for n in [10, 100, 1000, 10_000]:
    ops = approx_ops(n)
    print(f"n = {n}")
    for k, v in ops.items():
        print(f"  {k:10s} ~ {v}")
    print()

n = 10
  O(n)       ~ 10
  O(n log n) ~ 33
  O(n^2)     ~ 100
  O(2^n)     ~ 1024

n = 100
  O(n)       ~ 100
  O(n log n) ~ 664
  O(n^2)     ~ 10000
  O(2^n)     ~ None

n = 1000
  O(n)       ~ 1000
  O(n log n) ~ 9965
  O(n^2)     ~ 1000000
  O(2^n)     ~ None

n = 10000
  O(n)       ~ 10000
  O(n log n) ~ 132877
  O(n^2)     ~ 100000000
  O(2^n)     ~ None



## 9. Concluzii si idei de discutie

- Pentru fiecare problema am comparat solutii naive (`O(n^2)`, `O(n^3)`) cu solutii mai bune (`O(n log n)`, `O(n)`).
- Am folosit tehnici standard:
  - divide et impera (binary search, array rotit, exponentiere rapida),
  - prefix sums + hash map (k-sum subarray),
  - sliding window (probleme pe subarray/substring),
  - scheme de vot / counting (majority element),
  - sortare + doi pointeri (3-sum).
- Puncte de discutie la seminar:
  - de ce merita sa platim memorie suplimentara (hash map / set) pentru a reduce timpul?
  - cum recunoastem ca o problema se preteaza la sliding window?
  - cand este usor de trecut de la o solutie `O(n^2)` la una `O(n log n)` sau `O(n)` si cand nu.
